### Topic 7: Metadata Filtering

### Why Do We Need Metadata Filtering?

Suppose our Qdrant collection contains:

- FD FAQs
- Home Loan FAQs
- HR Policies
- Insurance Documents

A user asks:

> What is the premature withdrawal penalty?

Semantic search finds documents that are **similar in meaning**.

Without metadata filtering, Qdrant may return:

- FD FAQ
- HR Policy
- Insurance Document

Some results may be relevant, but they come from different document types.

---

### What Is Metadata Filtering?

Metadata filtering tells Qdrant **where to search** before performing semantic search.

For example: **Search only FAQs, Search only HR Policies, Search only Insurance Documents, or Search only one PDF.**

Instead of searching the entire collection, Qdrant searches only the filtered documents.

### Example

Suppose every document stores the following metadata:

```text
Document 1
Question : FD Interest Rate
doc_type : faq
source : fd_faq.pdf

Document 2
Policy : Employee Leave
doc_type : policy
source : hr_policy.pdf

Document 3
Question : Home Loan EMI
doc_type : faq
source : home_loan.pdf
```

A user asks:

> What is the FD interest rate?

If we apply:

```text
doc_type = faq
```

Qdrant searches only **Document 1** and **Document 3**. The HR policy document is ignored.

---

### Why Is Filtering Better?

Without filtering: **Search all documents, Mixed results, Slower retrieval.**

With filtering: **Search only relevant documents, Faster search, Better retrieval quality, More accurate answers.**

---

### Common Metadata Fields

**doc_type, source, language, product, department, country**

These fields are stored as **payload** when documents are inserted into Qdrant.

--- 
### Key Point

- **Semantic Search:** Which documents are most similar?
- **Metadata Filtering:** Which documents should be searched?

Both work together to retrieve the best results.

---

### Real-World Issues, Edge Cases & Debugging

**Wrong metadata field**

Problem: Filter uses `document_type` but the payload stores `doc_type`.

Result: Zero matching documents.

Solution: Ensure the filter field matches the payload key exactly.

---

**Wrong metadata value**

Problem: Filter uses `FAQ` but the payload stores `faq`.

Result: No documents are returned because matching is case-sensitive.

Solution: Use consistent values during ingestion and querying.

---

**Over-Filtering**

Problem: The filter is too restrictive, for example:

```text
doc_type = faq
source = fd_v1.pdf
language = Hindi
```

Result: Very few or no matching documents.

Solution: Apply only the filters required by the use case.

---

**Missing Metadata**

Problem: Some documents do not contain fields like `doc_type` or `source`.

Result: Those documents cannot match the filter.

Solution: Store consistent metadata for every document during ingestion.

---

**No Payload Index**

Problem: Filtering on a field without creating a payload index.

Result: Filtering still works but becomes slower as the collection grows.

Solution: Create payload indexes for frequently filtered fields.

---

### Common Mistakes

- Forgetting to store metadata during ingestion.
- Using incorrect payload field names.
- Using inconsistent metadata values (`FAQ` vs `faq`).
- Applying unnecessary filters that remove valid documents.
- Not creating payload indexes for commonly filtered fields.

---

### Lead-Level Interview Questions

**Q1. Why is metadata filtering important in a RAG application?**

**Answer**

Semantic search finds similar documents, while metadata filtering restricts the search to relevant documents such as a specific product, department, language, or source. This improves both retrieval speed and accuracy.

---

**Q2. Why is filtering inside Qdrant better than filtering in Python?**

**Answer**

Qdrant filters documents before similarity search, so only relevant documents are searched. Python filtering happens after retrieval, which wastes computation and may discard useful results.

---

**Q3. What metadata fields are commonly stored in Qdrant?**

**Answer**

Common fields include `doc_type`, `source`, `product`, `language`, `department`, `country`, and `version`.

---

**Q4. What happens if the metadata field name is incorrect?**

**Answer**

The filter does not match any documents, so Qdrant returns zero results. The payload key in the filter must exactly match the stored metadata.

---

**Q5. Does metadata filtering affect similarity search?**

**Answer**

Yes. Qdrant first applies the metadata filter and then performs similarity search only on the matching documents, reducing the search space and improving performance.

---

**Q6. Can a document have multiple metadata fields?**

**Answer**

Yes. A document can store multiple metadata fields such as `doc_type`, `product`, `language`, and `source`, and filters can combine them using AND, OR, and NOT conditions.

---

**Q7. When should you create a payload index?**

**Answer**

Create a payload index for metadata fields that are frequently used in filtering, such as `doc_type`, `source`, or `product`, especially in large collections.

---

In [1]:
"""
qdrant_filtering.py
---------------------
Metadata filtering in practice -- scoping search by doc_type and source.

Shows:
  - must (AND) filtering: only return chunks matching one doc_type
  - should (OR) filtering: return chunks from any of several doc_types
  - must_not (NOT) filtering: exclude a specific source from results
  - combined filtering: must + must_not together
  - the over-filtering edge case: filter leaves fewer points than k
  - how to debug a silent filter failure using client.retrieve()

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    PayloadSchemaType,
    Filter,
    FieldCondition,
    MatchValue,
    MatchAny,     # "field value must be one of this list" -- OR across values
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "fd_filtering_demo"
VECTOR_SIZE = 384
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

client = QdrantClient(":memory:")
model = SentenceTransformer(MODEL_NAME)

# corpus with a mix of doc_types and sources -- filtering is only useful when there's variety
CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
     "source": "FD_Policy_v2.json", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to death of the depositor.",
     "source": "FD_Policy_v2.json", "doc_type": "policy"},
    {"text": "DRAFT: Premature withdrawal penalty under review, may change to 0.5 percent.",
     "source": "FD_Policy_draft.json", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "What is the penalty for early FD closure? A 1 percent penalty applies.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "Can I break my FD before maturity? Yes, with a penalty on the interest rate.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "Step 1: Verify customer identity. Step 2: Check FD status before processing.",
     "source": "FD_SOP.json", "doc_type": "sop"},
]


def make_point_id(source: str, index: int) -> int:
    # deterministic ID so re-upserts update in place
    return int(hashlib.md5(f"{source}::{index}".encode()).hexdigest()[:8], 16)


def setup() -> None:
    # create collection and add payload index on doc_type
    existing = [c.name for c in client.get_collections().collections]
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )

    # payload index on doc_type -- makes filtered searches fast at scale
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="doc_type",
        field_schema=PayloadSchemaType.KEYWORD,
    )

    # also index source -- needed for source-level filtering later
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="source",
        field_schema=PayloadSchemaType.KEYWORD,
    )

    # embed and upsert all chunks
    texts = [c["text"] for c in CHUNKS]
    embeddings = model.encode(texts, normalize_embeddings=True)
    points = [
        PointStruct(
            id=make_point_id(CHUNKS[i]["source"], i),
            vector=embeddings[i].tolist(),
            payload={
                "text": CHUNKS[i]["text"],
                "source": CHUNKS[i]["source"],
                "doc_type": CHUNKS[i]["doc_type"],
            },
        )
        for i in range(len(CHUNKS))
    ]
    client.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"Setup complete: {client.get_collection(COLLECTION_NAME).points_count} points\n")


def search(query: str, k: int = 3, query_filter=None, label: str = "") -> None:
    """Generic search helper -- reused by all demo functions below."""
    query_vector = model.encode(query, normalize_embeddings=True).tolist()
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,  # None = no filter = search everything
        limit=k,
        with_payload=True,
    ).points

    print(f"--- {label} (k={k}, got={len(results)}) ---")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] [{r.payload['source']}]")
        print(f"           {r.payload['text'][:70]}")
    print()


QUERY = "What is the penalty for closing an FD early?"


def demo_no_filter() -> None:
    """No filter -- searches all 8 chunks across all doc types."""
    search(QUERY, k=4, query_filter=None, label="No filter (all doc types)")


def demo_must_single() -> None:
    """must with one condition: only return FAQ chunks.
    The k results are all FAQs -- no post-search discarding needed."""
    search(
        QUERY, k=3,
        query_filter=Filter(
            must=[
                # must = ALL conditions in this list must be true (AND logic)
                FieldCondition(key="doc_type", match=MatchValue(value="faq"))
            ]
        ),
        label="Filter: doc_type must be 'faq'",
    )


def demo_should_or() -> None:
    """should with two conditions: return FAQ OR policy chunks.
    At least one of the should conditions must match."""
    search(
        QUERY, k=4,
        query_filter=Filter(
            should=[
                # should = at least one of these must be true (OR logic)
                FieldCondition(key="doc_type", match=MatchValue(value="faq")),
                FieldCondition(key="doc_type", match=MatchValue(value="policy")),
            ]
        ),
        label="Filter: doc_type should be 'faq' OR 'policy'",
    )


def demo_match_any() -> None:
    """MatchAny is a cleaner way to express the same OR across values."""
    search(
        QUERY, k=4,
        query_filter=Filter(
            must=[
                # MatchAny = field value must be one of these -- equivalent to the should above
                FieldCondition(key="doc_type", match=MatchAny(any=["faq", "policy"]))
            ]
        ),
        label="Filter: doc_type in ['faq', 'policy'] using MatchAny",
    )


def demo_must_not() -> None:
    """must_not: exclude the draft source entirely.
    Useful for version-aware retrieval -- only search approved documents."""
    search(
        QUERY, k=4,
        query_filter=Filter(
            must_not=[
                # must_not = this condition must be FALSE (NOT logic)
                FieldCondition(key="source", match=MatchValue(value="FD_Policy_draft.json"))
            ]
        ),
        label="Filter: exclude draft source (must_not)",
    )


def demo_combined() -> None:
    """Combined must + must_not: only policy docs, but not the draft version.
    This is the "only search current approved policy" pattern."""
    search(
        QUERY, k=3,
        query_filter=Filter(
            must=[
                # only policy documents
                FieldCondition(key="doc_type", match=MatchValue(value="policy"))
            ],
            must_not=[
                # but not the draft
                FieldCondition(key="source", match=MatchValue(value="FD_Policy_draft.json"))
            ],
        ),
        label="Filter: policy only, exclude draft (must + must_not)",
    )


def demo_over_filtering() -> None:
    """Over-filtering edge case: SOP doc_type has only 1 chunk in the collection.
    Requesting k=3 returns only 1 result -- not an error, just correct behavior.
    Code that assumes len(results) == k will break here."""
    search(
        QUERY, k=3,
        query_filter=Filter(
            must=[FieldCondition(key="doc_type", match=MatchValue(value="sop"))]
        ),
        label="Over-filtering: only 1 SOP chunk exists but k=3 requested",
    )


def demo_debug_silent_failure() -> None:
    """How to debug a silent filter failure -- wrong key name returns zero results.
    Use client.retrieve() to see exactly what's stored before assuming the filter is right."""

    print("--- Debug: wrong payload key name (silent failure) ---")
    # this filter uses 'document_type' but the payload stores 'doc_type' -- returns nothing
    query_vector = model.encode(QUERY, normalize_embeddings=True).tolist()
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        # wrong key name -- will silently return zero results, no error raised
        query_filter=Filter(
            must=[FieldCondition(key="document_type", match=MatchValue(value="faq"))]
        ),
        limit=3,
        with_payload=True,
    ).points
    print(f"  Results with wrong key 'document_type': {len(results)} (expected 0)")

    # fix: use client.retrieve() to inspect what's actually stored in the payload
    all_ids = [make_point_id(CHUNKS[i]["source"], i) for i in range(len(CHUNKS))]
    sample = client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=[all_ids[0]],  # look at the first point
        with_payload=True,
        with_vectors=False,
    )
    print(f"  Actual payload keys stored: {list(sample[0].payload.keys())}")
    print(f"  Use these exact key names in your filter conditions\n")


# ======================================================================
# Run all demos in order
# ======================================================================

setup()
demo_no_filter()
demo_must_single()
demo_should_or()
demo_match_any()
demo_must_not()
demo_combined()
demo_over_filtering()
demo_debug_silent_failure()

C:\Users\pauls\AppData\Local\Temp\ipykernel_52452\2066581024.py:77: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


Setup complete: 8 points

--- No filter (all doc types) (k=4, got=4) ---
  score=0.8909 | [faq] [FD_FAQ.json]
           What is the penalty for early FD closure? A 1 percent penalty applies.
  score=0.6248 | [faq] [FD_FAQ.json]
           Can I break my FD before maturity? Yes, with a penalty on the interest
  score=0.5689 | [policy] [FD_Policy_v2.json]
           Premature withdrawal incurs a 1 percent penalty on the applicable rate
  score=0.5678 | [policy] [FD_Policy_v2.json]
           This penalty does not apply if the FD is closed due to death of the de

--- Filter: doc_type must be 'faq' (k=3, got=2) ---
  score=0.8909 | [faq] [FD_FAQ.json]
           What is the penalty for early FD closure? A 1 percent penalty applies.
  score=0.6248 | [faq] [FD_FAQ.json]
           Can I break my FD before maturity? Yes, with a penalty on the interest

--- Filter: doc_type should be 'faq' OR 'policy' (k=4, got=4) ---
  score=0.8909 | [faq] [FD_FAQ.json]
           What is the penalty for ear